In [ ]:
# If needed, uncomment:
# !pip install -q pandas numpy matplotlib seaborn

import warnings

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

warnings.filterwarnings("ignore")

# ================================================================
# 1. LOAD DATA AND SAFE COLUMN DETECTION
# ================================================================
file_path = "Table1.csv"

try:
    df = pd.read_csv(file_path, low_memory=False)
    print(f"File loaded successfully: {file_path}")
    print(f"Rows: {df.shape[0]} | Columns: {df.shape[1]}")
except FileNotFoundError:
    print(f"Error: the file {file_path} was not found.")
    raise

race_columns = [
    c for c in df.columns
    if "raça" in c.lower() or "raca" in c.lower() or "etnia" in c.lower()
]

if not race_columns:
    print("ERROR: No Race/Ethnicity column was found in the CSV file.")
else:
    col_race = race_columns[0]

    infectious_cols = [
        c for c in df.columns
        if "doenças infecciosas" in c.lower()
        and "choice=" in c.lower()
        and "não" not in c.lower()
    ]

    chronic_cols = [
        c for c in df.columns
        if "doenças crônicas" in c.lower()
        and "choice=" in c.lower()
        and "não" not in c.lower()
    ]

    ciap_cols = [c for c in df.columns if "ciap" in c.lower()]

    # ============================================================
    # 2. TRANSLATION MAPS
    # ============================================================
    ciap_translation = {
        "K86-HIPERTENSÃO SEM COMPLICAÇÕES": "K86-HYPERTENSION WITHOUT COMPLICATIONS",
        "P01-SENSAÇÃO DE ANSIEDADE/NERVOSISMO/TENSÃO": "P01-FEELING ANXIOUS/NERVOUS/TENSE",
        "P06-PERTURBAÇÃO DO SONO": "P06-SLEEP DISTURBANCE",
        "R05-TOSSE": "R05-COUGH",
        "N01-CEFALÉIA": "N01-HEADACHE",
        "P19-ABUSO DE DROGAS": "P19-DRUG ABUSE",
        "L18-DORES MUSCULARES": "L18-MUSCLE PAIN",
        "K87-HIPERTENSÃO COM COMPLICAÇÕES": "K87-HYPERTENSION WITH COMPLICATIONS",
        "T90-DIABETES NÃO INSULINO-DEPENDENTE": "T90-NON-INSULIN-DEPENDENT DIABETES",
        "R74-INFECÇÃO AGUDA DO TRATO RESPIRATÓRIO SUPERIOR": "R74-ACUTE UPPER RESPIRATORY TRACT INFECTION",
        "A04-FRAQUEZA/CANSAÇO GERAL": "A04-GENERAL WEAKNESS/FATIGUE",
        "A03-FEBRE": "A03-FEVER",
        "D01-DOR ABDOMINAL/CÓLICA": "D01-ABDOMINAL PAIN/CRAMPS",
        "L03-SINTOMAS/QUEIXAS LOMBARES": "L03-LOW BACK SYMPTOMS/COMPLAINTS",
        "P76-DEPRESSÃO": "P76-DEPRESSION",
        "R02-FALTA DE AR/DISPNEIA": "R02-SHORTNESS OF BREATH/DYSPNEA",
    }

    def translate_ciap_term(value):
        value_str = str(value).strip()
        return ciap_translation.get(value_str, value_str)

    # ============================================================
    # 3. LONGITUDINAL PROCESSING (FORWARD FILL) AND IBGE MAPPING
    # ============================================================
    df = df.sort_values(["Record ID", "Repeat Instance"], na_position="first")
    cols_to_fill = [col_race] + infectious_cols + chronic_cols
    cols_to_fill = [c for c in cols_to_fill if c in df.columns]

    df[cols_to_fill] = df.groupby("Record ID")[cols_to_fill].ffill()

    df_visits = df[df["Repeat Instrument"].notna()].copy()
    df_visits = df_visits.dropna(subset=[col_race])

    def map_race_flexible(value):
        v = str(value).lower().strip()
        if "branca" in v or "branco" in v:
            return "White"
        if "preta" in v:
            return "Black"
        if "parda" in v or "pardo" in v:
            return "Brown"
        if "amarela" in v or "amarelo" in v:
            return "Asian"
        if "indígena" in v or "indigena" in v:
            return "Indigenous"
        return None

    df_visits["Race_Group"] = df_visits[col_race].apply(map_race_flexible)
    df_plot = df_visits.dropna(subset=["Race_Group"]).copy()

    # ============================================================
    # 4. SEPARATE DISEASE BURDEN CALCULATION
    # ============================================================
    def to_binary(value):
        return 1 if str(value).lower().strip() in [
            "checked", "1", "1.0", "sim", "verdadeiro"
        ] else 0

    for c in infectious_cols + chronic_cols:
        df_plot[c] = df_plot[c].apply(to_binary).astype(int)

    df_plot["Infectious Burden"] = df_plot[infectious_cols].sum(axis=1)
    df_plot["Chronic Burden"] = df_plot[chronic_cols].sum(axis=1)

    # ============================================================
    # 5. PREPARATION (MELT) TO COMPARE DISEASE TYPES
    # ============================================================
    df_melted = df_plot.melt(
        id_vars=["Record ID", "Race_Group"],
        value_vars=["Infectious Burden", "Chronic Burden"],
        var_name="Disease Type",
        value_name="Number of Diseases",
    )

    race_order = [
        g for g in ["White", "Brown", "Black", "Asian", "Indigenous"]
        if g in df_plot["Race_Group"].unique()
    ]

    # ============================================================
    # 6. GENERATE AND EXPORT ARTICLE TABLES (CSV)
    # ============================================================
    burden_table = (
        df_melted.groupby(["Race_Group", "Disease Type"])["Number of Diseases"]
        .agg(["mean", "std", "count"])
        .reset_index()
    )
    burden_table.columns = [
        "Race/Color (IBGE)",
        "Clinical Profile",
        "Mean Number of Diseases",
        "Standard Deviation",
        "N (Sample)",
    ]

    burden_table["Race/Color (IBGE)"] = pd.Categorical(
        burden_table["Race/Color (IBGE)"],
        categories=race_order,
        ordered=True
    )
    burden_table = burden_table.sort_values(["Race/Color (IBGE)", "Clinical Profile"])

    burden_file = "race_disease_burden_base_en.csv"
    burden_table.to_csv(burden_file, index=False, encoding="utf-8-sig")

    ciap_rows = []
    ciap_file = None

    if ciap_cols:
        target_ciap_col = ciap_cols[0]

        for group in race_order:
            subgroup = df_plot[df_plot["Race_Group"] == group]
            top_ciaps = subgroup[target_ciap_col].dropna().value_counts().head(5)

            for ciap, count in top_ciaps.items():
                ciap_rows.append(
                    {
                        "Race/Color (IBGE)": group,
                        "Total N (Group)": len(subgroup),
                        "CIAP-2 Reason": translate_ciap_term(ciap),
                        "Frequency": count,
                        "Proportion (%)": round((count / len(subgroup) * 100), 1) if len(subgroup) > 0 else 0,
                    }
                )

        ciap_table = pd.DataFrame(ciap_rows)
        ciap_file = "ciap_by_race_en.csv"
        ciap_table.to_csv(ciap_file, index=False, encoding="utf-8-sig")

    print(f"\n{'=' * 80}")
    print("DATA TABLES EXPORTED SUCCESSFULLY:")
    print(f"- '{burden_file}' (chart statistics)")
    if ciap_file:
        print(f"- '{ciap_file}' (consultation reasons)")
    print(f"{'=' * 80}")

    # ============================================================
    # 7. CONSOLE REPORT (NUMERIC SUMMARY AND CIAP-2)
    # ============================================================
    print(f"\n{'=' * 80}")
    print("NUMERIC SUMMARY: MEAN NUMBER OF DISEASES BY GROUP")
    print(f"{'=' * 80}")
    group_means = df_plot.groupby("Race_Group")[["Infectious Burden", "Chronic Burden"]].mean()
    print(group_means.reindex(race_order).round(3))

    print(f"\n{'=' * 80}")
    print("MAIN REASONS FOR CONSULTATION (CIAP-2) BY RACE/COLOR")
    print(f"{'=' * 80}")

    if not ciap_cols:
        print("No column containing the term 'CIAP' was found.")
    else:
        for group in race_order:
            subgroup = df_plot[df_plot["Race_Group"] == group]
            print(f"\n[{group.upper()}] - Visits analyzed: {len(subgroup)}")
            if len(subgroup) > 0:
                top_ciaps = subgroup[target_ciap_col].dropna().value_counts().head(3)
                if len(top_ciaps) > 0:
                    for ciap, count in top_ciaps.items():
                        ciap_en = translate_ciap_term(ciap)
                        print(f"  -> {ciap_en[:60]:<60} ({count} cases)")
                else:
                    print("  -> No CIAP registered.")
    print(f"\n{'=' * 80}")

    # ============================================================
    # 8. VISUALIZATION — GROUPED BARPLOT
    # ============================================================
    plt.figure(figsize=(12, 8))
    sns.set_theme(style="whitegrid")

    disease_palette = {
        "Infectious Burden": "#e74c3c",
        "Chronic Burden": "#2980b9",
    }

    sns.barplot(
        data=df_melted,
        x="Race_Group",
        y="Number of Diseases",
        hue="Disease Type",
        palette=disease_palette,
        capsize=0.05,
        err_kws={"linewidth": 1.5},
        order=race_order,
    )

    plt.title(
        "Comparison: Infectious vs. Chronic Disease Burden by Race/Color",
        fontsize=16,
        fontweight="bold",
        pad=20,
    )
    plt.ylabel(
        "Mean Number of Diseases per Patient (with 95% CI)",
        fontsize=13,
        fontweight="bold",
    )
    plt.xlabel(
        "Race / Color (IBGE Standard)",
        fontsize=13,
        fontweight="bold",
    )

    sns.despine(left=True, bottom=True)
    plt.legend(
        title="Clinical Profile Type",
        bbox_to_anchor=(1.02, 1),
        loc="upper left",
        frameon=False,
        fontsize=12,
    )

    plt.tight_layout()
    plt.show()